# Freqtrade in Google Colab

Dieses Notebook richtet eine voll funktionsfähige Freqtrade-Umgebung in Google Colab ein und zeigt Schritt für Schritt, wie du den Bot für Backtests, Optimierungen und den Paper-Trading-Betrieb verwendest.

## 1. Umgebung initialisieren

Führe die folgende Zelle aus, um alle Abhängigkeiten zu installieren und die Freqtrade-Verzeichnisstruktur anzulegen.

In [ ]:
%%bash
set -e
python -m pip install --quiet --upgrade pip
python -m pip install --quiet "freqtrade==2024.6" "ccxt>=4.0.0" plotly taipy
freqtrade create-userdir --userdir /content/freqtrade


## 2. Optional: Google Drive einbinden

Wenn du Konfigurationen oder Ergebnisse dauerhaft speichern möchtest, mounte deinen Google Drive und verschiebe das Arbeitsverzeichnis nach `/content/drive/MyDrive`.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
base_path = Path('/content/drive/MyDrive/freqtrade')
base_path.mkdir(parents=True, exist_ok=True)
!rm -rf /content/freqtrade
!ln -s $base_path /content/freqtrade
print('Arbeitsverzeichnis:', base_path)


## 3. Konfiguration und Beispiel-Strategie

Die nächste Zelle erzeugt eine minimalistische Beispiel-Strategie sowie eine Konfigurationsdatei mit sinnvollen Standardeinstellungen für Backtests und Paper-Trading.

In [ ]:
from pathlib import Path
import json

userdir = Path('/content/freqtrade')
(userdir / 'user_data/strategies').mkdir(parents=True, exist_ok=True)

strategy = '''
from freqtrade.strategy import IStrategy
from pandas import DataFrame

class ColabExampleStrategy(IStrategy):
    timeframe = '1h'
    startup_candle_count = 50
    minimal_roi = {"0": 0.04, "120": 0.02, "240": 0.01, "720": 0}
    stoploss = -0.1

    def populate_indicators(self, dataframe: DataFrame, metadata: dict) -> DataFrame:
        dataframe['ema_short'] = dataframe['close'].ewm(span=12).mean()
        dataframe['ema_long'] = dataframe['close'].ewm(span=26).mean()
        return dataframe

    def populate_entry_trend(self, dataframe: DataFrame, metadata: dict) -> DataFrame:
        dataframe.loc[
            (dataframe['ema_short'] > dataframe['ema_long']) &
            (dataframe['volume'] > 0),
            'enter_long'
        ] = 1
        return dataframe

    def populate_exit_trend(self, dataframe: DataFrame, metadata: dict) -> DataFrame:
        dataframe.loc[
            (dataframe['ema_short'] < dataframe['ema_long']),
            'exit_long'
        ] = 1
        return dataframe
'''

(userdir / 'user_data/strategies/ColabExampleStrategy.py').write_text(strategy)

config = {
    "user_data_dir": str(userdir),
    "dataformat_ohlcv": "json",
    "dry_run": True,
    "dry_run_wallet": 1000,
    "stake_currency": "USDT",
    "stake_amount": "unlimited",
    "exchange": {
        "name": "binance",
        "key": "",
        "secret": "",
        "ccxt_config": {},
        "pair_whitelist": ["BTC/USDT", "ETH/USDT", "SOL/USDT"],
        "pair_blacklist": []
    },
    "timeframe": "1h",
    "max_open_trades": 3,
    "tradable_balance_ratio": 0.99,
    "stoploss": -0.1,
    "strategy": "ColabExampleStrategy",
    "forcebuy_enable": True,
    "timeframe_detail": "5m",
    "telegram": {
        "enabled": False
    },
    "api_server": {
        "enabled": False,
        "listen_ip_address": "0.0.0.0",
        "listen_port": 8080,
        "verbosity": "info",
        "jwt_secret_key": "changeme",
        "CORS_origins": []
    }
}

(userdir / 'user_data/config.json').write_text(json.dumps(config, indent=4))
print('Strategie und Konfigurationsdatei wurden erstellt.')


## 4. Marktdaten herunterladen

Lade historische Daten über die integrierte Download-Funktion. Wähle dazu Exchange, Paar(e) und Zeitrahmen passend zu deiner Strategie.

In [ ]:
!freqtrade download-data     --userdir /content/freqtrade     --exchange binance     --timeframes 1h 5m     --days 30     --pairs BTC/USDT ETH/USDT SOL/USDT


## 5. Backtesting

Mit der folgenden Zelle führst du einen Backtest gegen die heruntergeladenen Daten aus und erhältst eine Zusammenfassung der Performance.

In [ ]:
!freqtrade backtesting     --userdir /content/freqtrade     --config /content/freqtrade/user_data/config.json     --strategy ColabExampleStrategy     --timeframe 1h


## 6. Hyperparameter-Optimierung (Hyperopt)

Nutze Hyperopt, um Strategie-Parameter automatisch zu optimieren. Passe `--spaces` und die Anzahl der Epochen an deine Bedürfnisse an.

In [ ]:
!freqtrade hyperopt     --userdir /content/freqtrade     --config /content/freqtrade/user_data/config.json     --strategy ColabExampleStrategy     --hyperopt-loss SharpeHyperOptLossDaily     --spaces buy sell roi stoploss     --epochs 100


## 7. Paper-Trading / Dry-Run starten

Aktiviere den Bot im Dry-Run-Modus, um Trades ohne echtes Kapital zu simulieren. In Colab empfiehlt sich die Verwendung eines separaten Tabs oder `tmux` via `colab_ssh`, damit der Prozess dauerhaft laufen kann.

In [ ]:
!freqtrade trade     --userdir /content/freqtrade     --config /content/freqtrade/user_data/config.json     --strategy ColabExampleStrategy     --dry-run


## 8. Nützliche Tipps

- Passe die `pair_whitelist` und `timeframe` an dein Portfolio und deinen Handelsstil an.
- Nutze das `freqtrade plot-dataframe` Kommando, um Candle-Charts inklusive Indikatoren zu visualisieren.
- Speichere Ergebnisse (Backtests, Hyperopt-Resultate) in Google Drive, um sie außerhalb der aktuellen Session verfügbar zu machen.
- Verwende `freqtrade list-pairs` oder `freqtrade list-timeframes`, um unterstützte Märkte und Zeitfenster deines Exchanges zu prüfen.